# Main demos for datadownload 

In [ ]:
#Temporary prior to pip installing the library
import sys
sys.path.append("..")
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
#from hrud import HRUD
import numpy as np
import hrud as hd
from sklearn.neighbors import KDTree, BallTree

In [ ]:
%matplotlib inline

## Setup the library (run the constructor to get acces to some general parameters)

In [ ]:
hd = hd.HRUD()

## Downloading data for different cities and plotting

In [ ]:
lima = hd.download_osm(2, 'Lima,Peru')
quito = hd.download_osm(1, 'Quito, Ecuador')
ba = hd.download_osm(0, 'Buenos Aires, Argentina')

In [ ]:
lima.plot()
plt.xlabel('lon')
plt.ylabel('lat')
plt.title('Lima')
plt.show()

quito.plot()
plt.xlabel('lon')
plt.ylabel('lat')
plt.title('Quito')
plt.show()

ba.plot()
plt.xlabel('lon')
plt.ylabel('lat')
plt.title('Buenos Aires')
plt.show()

Lima is still missing the Callao polygon, lets download it as well

In [ ]:
callao = hd.download_osm(1, 'Callao, Peru')

### Downloading population data

In [ ]:
pop_per = hd.download_hdx("4e74db39-87f1-4383-9255-eaf8ebceb0c9/resource/317f1c39-8417-4bde-a076-99bd37feefce/download/population_per_2018-10-01.csv.zip")
pop_ecu = hd.download_hdx('58c3ac3f-febd-4222-8969-59c0fe0e7a0d/resource/c05a3c81-a78c-4e6c-ac05-de1316d4ba12/download/population_ecu_2018-10-01.csv.zip')
pop_arg = hd.download_hdx('6cf49080-1226-4eda-8700-a0093cbdfe4d/resource/5737d87f-e17f-4c82-b1bd-d589ed631318/download/population_arg_2018-10-01.csv.zip')

In [ ]:
pop_per.head()

### Merging layers

In [ ]:
lima = hd.merge_geom_downloads([lima, callao])

### Conversion to Point geometries and hexagons

We got the lat lon coordinates but their are just in numerical format, we need them as geometries to perform spatial operations on the hexagon grid

First, lets filter the national population to our city bounds

In [ ]:
pop_lima = hd.filter_population(pop_per, lima)
pop_quito = hd.filter_population(pop_ecu, quito)
pop_ba = hd.filter_population(pop_arg, ba)

Applying filter_population gives us the set of points within our city's bounds and a Point geometry to work with hexagons

In [ ]:
pop_lima.head()

### Removing unnecesary features

We don't need the San Lorenzo island. By providing a bounding box we an remove it

In [ ]:
pop_lima = hd.remove_features(pop_lima, [-12.2,-12, -77.3,-77.17])

Let's generate the hexagon grid using Uber's H3

In [ ]:
hex_lima, hex_lima_centroids = hd.gen_hexagons(8, lima)
hex_quito, hex_quito_centroids = hd.gen_hexagons(10, quito)
hex_ba, hex_ba_centroids = hd.gen_hexagons(9, ba)

Notice the effect of the resolution parameter (higher values apply generate smaller hexagons)

In [ ]:
print(hex_lima.shape)
print(hex_quito.shape)
print(hex_ba.shape)

### Merging a layer

We got both the population point geometries and hexagons, lets get the population per hexagon

In [ ]:
hex_lima = hd.merge_shape_hex(hex_lima, 
                   pop_lima, 
                   how='inner', 
                   op='intersects', 
                   agg={'population_2020': 'sum', 'population_2015':'mean'})

hex_quito = hd.merge_shape_hex(hex_quito, 
                   pop_quito, 
                   how='inner', 
                   op='intersects', 
                   agg={'population_2020': 'sum', 'population_2015':'mean'})

hex_ba = hd.merge_shape_hex(hex_ba, 
                   pop_ba, 
                   how='inner', 
                   op='intersects', 
                   agg={'population_2020': 'sum', 'population_2015':'mean'})

We've used an inner join to keep only the points that intersect a hexagon (in this case, a within operation gives similar results). We provide a dictionary detailing how to aggregate our fields of interest <br>

Let's visualize the maps

In [ ]:
hex_lima.plot(column='population_2020', legend=True, cmap='viridis', figsize=(10,5), missing_kwds={'color':'grey'})
plt.xlabel('lon')
plt.ylabel('lat')
plt.title('Lima')
plt.show

hex_quito.plot(column='population_2020', legend=True, cmap='viridis', figsize=(10,5), missing_kwds={'color':'grey'})
plt.xlabel('lon')
plt.ylabel('lat')
plt.title('Quito')
plt.show

hex_ba.plot(column='population_2020', legend=True, cmap='viridis', figsize=(10,5), missing_kwds={'color':'grey'})
plt.xlabel('lon')
plt.ylabel('lat')
plt.title('Buenos Aires')
plt.show

In [ ]:
!ls ../merge_inei2017/lima_departamento_zonas

# Lets map our census zones in PyDeck

In [ ]:
zonas = gpd.read_file('../merge_inei2017/lima_departamento_zonas')

In [ ]:
filter_zonas = hd.filter_population(pop_per, zonas)

In [ ]:
filter_zonas.head()

In [ ]:
zonas = hd.merge_shape_hex(zonas, filter_zonas, how='inner', op='within', agg={'population_2020': 'sum'})

In [ ]:
centroids = zonas.geometry.centroid

lat = centroids.y
lon = centroids.x

b = zonas.bounds

bounds = [b['minx'].min(),
          b['miny'].min(),
          b['maxx'].max(),
          b['maxy'].max()]

fs = hd.download_overpass_poi(bounds, 'food_supply')

from math import radians

fs['lat_radians'] = fs['lat'].apply(radians)
fs['lon_radians'] = fs['lon'].apply(radians)

points = np.array(list(zip(lat, lon)))

points = pd.DataFrame(points, columns=['lat', 'lon'])
points['lat_radians'] = points['lat'].apply(radians)
points['lon_radians'] = points['lon'].apply(radians)

In [ ]:
tree = BallTree(fs[['lat_radians', 'lon_radians']].values, metric='haversine')

In [ ]:
tree.query(points[['lat_radians', 'lon_radians']].values)[0] * 6371000/1000

In [ ]:
distances = tree.query(points[['lat_radians', 'lon_radians']].values)[0] * 6371000/1000

In [ ]:
walking_time = distances / 5

In [ ]:
zonas['dist_clostest_fs'] = distances

zonas['time_closest_fs'] = walking_time

In [ ]:
lima.crs = 'EPSG:4326'

In [ ]:
zonas.plot(figsize=(20,10), column='time_closest_fs', legend=True, missing_kwds={'color':'grey'})

In [ ]:
import plotly.graph_objects as go

In [ ]:
fig = go.Figure(
    go.Choroplethmapbox(
        geojson=zonas[['geometry']].__geo_interface__,
        locations=zonas.index.values.tolist(),
        #z=np.ones(shape=lima_metropolitana_h.shape[0]), # Just one color
        z=zonas['population_2020'].values.tolist(),
        showscale=False,
        marker_opacity=0.5,
        marker_line_width=0,
    )
)
fig.update_layout(
    mapbox_style="carto-positron", 
    mapbox_zoom=9, 
    mapbox_pitch=60, 
    mapbox_bearing=0, 
    mapbox_center = {"lat": zonas.centroid.y[0], 
                     "lon": zonas.centroid.x[0]})

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [ ]:
import pydeck as pdk

In [ ]:
zonas['dist_closest_fs'] = zonas['dist_clostest_fs'] * 1000
zonas['time_closest_fs'] = zonas['time_closest_fs'] * 60

In [ ]:
zonas.to_file('zonas_censales.geojson', driver='GeoJSON')

In [ ]:
json = zonas.__geo_interface__

In [ ]:
json.keys()

In [ ]:
def swap_xy(geom):
    if geom.is_empty:
        return geom

    if geom.has_z:
        def swap_xy_coords(coords):
            for x, y, z in coords:
                yield (y, x, z)
    else:
        def swap_xy_coords(coords):
            for x, y in coords:
                yield (y, x)

    # Process coordinates from each supported geometry type
    if geom.type in ('Point', 'LineString', 'LinearRing'):
        return type(geom)(list(swap_xy_coords(geom.coords)))
    elif geom.type == 'Polygon':
        ring = geom.exterior
        shell = type(ring)(list(swap_xy_coords(ring.coords)))
        holes = list(geom.interiors)
        for pos, ring in enumerate(holes):
            holes[pos] = type(ring)(list(swap_xy_coords(ring.coords)))
        return type(geom)(shell, holes)
    elif geom.type.startswith('Multi') or geom.type == 'GeometryCollection':
        # Recursive call
        return type(geom)([swap_xy(part) for part in geom.geoms])
    else:
        raise ValueError('Type %r not recognized' % geom.type)

In [ ]:
json = zonas.__geo_interface__

In [ ]:
from numba import jit

In [ ]:
@jit
def tuples_to_lists(json):
    for i in range(len(json['features'])):
        t = [list(x) for x in json['features'][i]['geometry']['coordinates']]
        poly = [[list(x) for x in t[0]]]
        
        json['features'][i]['geometry']['coordinates'] = poly

In [ ]:
tuples_to_lists(json)

In [ ]:
del json['bbox']

In [ ]:
zonas.head()

In [ ]:
df = pd.DataFrame.from_dict(json)

In [ ]:
polygon_df = pd.DataFrame()
polygon_df['coordinates'] = df['features'].apply(lambda row: row['geometry']['coordinates'])
polygon_df['population_2020'] = df['features'].apply(lambda row: row['properties']['population_2020'])
polygon_df['time_closest_fs'] = df['features'].apply(lambda row: row['properties']['time_closest_fs'])
polygon_df.head()

In [ ]:
cmap = plt.get_cmap(name='magma')

polygon_df['time_bins'] = pd.cut(
    polygon_df['time_closest_fs'], 
    bins = [0, 15, 30, 45, 60, 90]
)

bins_labels = polygon_df['time_bins'].unique().categories.values
bins_replace = {label: i  for i, label in enumerate(bins_labels)}
polygon_df['count'] = polygon_df['time_bins'].replace(bins_replace)

cmap_ixs = [int(round(255 / (i+1))) for i in polygon_df['count'].fillna(0).values]
rgb_values = [[color * 255 for color in cmap.colors[cmap_ix]] for cmap_ix in cmap_ixs]
rgb_count_df = pd.DataFrame(rgb_values, columns=['r','g','b'])

polygon_df = pd.concat((polygon_df, rgb_count_df), axis=1) 

polygon_df['r'] = polygon_df['r'].round(0)
polygon_df['g'] = polygon_df['g'].round(0)
polygon_df['b'] = polygon_df['b'].round(0)

In [ ]:
polygon_df

In [ ]:
INITIAL_VIEW_STATE = pdk.ViewState(
  latitude=-12,
  longitude=-77,
  zoom=11,
  max_zoom=16,
  pitch=45,
  bearing=0
)

geojson_ = pdk.Layer(
    'PolygonLayer',
    polygon_df[['coordinates', 'population_2020', 'r', 'g', 'b', 'time_closest_fs']],
    get_polygon='coordinates',
    opacity=0.8,
    stroked=False,
    filled=True,
    extruded=True,
    wireframe=True,
    get_elevation='population_2020 / 20',
    get_fill_color='[r, g, b]',
    get_line_color=[255, 255, 255],
    pickable=True,
    auto_highlight=True
)


tooltip = """
<b>Polygon info</b>:<br>
<b>Duración</b>: {time_closest_fs} minutos <br>
<b>Población 2020</b>: {population_2020} <br>
--------------------------------- <br>
"""

r = pdk.Deck(
    layers=[geojson_],
    initial_view_state=INITIAL_VIEW_STATE,
    tooltip={"html":tooltip},
    map_style='mapbox://styles/havb/ck8ajsl0t14m01ipfoizwegja', 
    mapbox_key='***REMOVED***'
)


r.to_html('zonas_censales_pob2020_TEST.html')

# Using HRUD to generate the same map

In [ ]:
import sys
sys.path.append("..")
import geopandas as gpd
import numpy as np
import pandas as pd
from hrud import HRUD
from math import radians

In [ ]:
hd = HRUD()

In [ ]:
zonas = gpd.read_file('./zonas_censales.geojson') 

In [ ]:
centroids = zonas.geometry.centroid
lat = centroids.y
lon = centroids.x

b = zonas.bounds
bounds = [b['minx'].min(),
          b['miny'].min(),
          b['maxx'].max(),
          b['maxy'].max()]

fs = hd.download_overpass_poi(bounds, 'food_supply')
fs['lat_radians'] = fs['lat'].apply(radians)
fs['lon_radians'] = fs['lon'].apply(radians)

points = np.array(list(zip(lat, lon)))
points = pd.DataFrame(points, columns=['lat', 'lon'])
points['lat_radians'] = points['lat'].apply(radians)
points['lon_radians'] = points['lon'].apply(radians)

Neighborhood search

In [ ]:
dist_hrud = hd.nn_search(fs[['lat_radians', 'lon_radians']].values, 
             points[['lat_radians', 'lon_radians']].values, 
             metric='haversine')

Verify we get the exact same distances and neighbors

In [ ]:
time_hrud = dist_hrud/5

In [ ]:
zonas['dist_clostest_fs_hrud'] = dist_hrud * 1000
zonas['time_closest_fs_hrud'] = time_hrud *60

In [ ]:
plot_df = hd.pydeck_df(zonas, 
             ['population_2020', 'time_closest_fs_hrud'], 
             'magma', 
             [0, 15, 30, 45, 60, 90], 
             'time_closest_fs_hrud')

In [ ]:
import pydeck as pdk

In [ ]:
INITIAL_VIEW_STATE = pdk.ViewState(
  latitude=-12,
  longitude=-77,
  zoom=11,
  max_zoom=16,
  pitch=45,
  bearing=0
)

geojson_ = pdk.Layer(
    'PolygonLayer',
    plot_df[['coordinates', 'population_2020', 'r', 'g', 'b', 'time_closest_fs_hrud']],
    get_polygon='coordinates',
    opacity=0.8,
    stroked=False,
    filled=True,
    extruded=True,
    wireframe=True,
    get_elevation='population_2020 / 20',
    get_fill_color='[r, g, b]',
    get_line_color=[255, 255, 255],
    pickable=True,
    auto_highlight=True
)


tooltip = """
<b>Polygon info</b>:<br>
<b>Duración</b>: {time_closest_fs} minutos <br>
<b>Población 2020</b>: {population_2020} <br>
--------------------------------- <br>
"""

r = pdk.Deck(
    layers=[geojson_],
    initial_view_state=INITIAL_VIEW_STATE,
    tooltip={"html":tooltip},
    map_style='mapbox://styles/havb/ck8ajsl0t14m01ipfoizwegja', 
    mapbox_key='***REMOVED***'
)


r.to_html('zonas_censales_pob2020_TEST.html')